In [1]:
from os import PathLike
from hdfs import InsecureClient
from pyspark.sql import SparkSession
from pyspark.sql import Row
from delta import *
from pyspark.sql.types import LongType, StringType, StructField, StructType, BooleanType, ArrayType, IntegerType, DoubleType

In [2]:
# warehouse_location points to the default location for managed databases and tables
warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

builder = SparkSession \
    .builder \
    .appName("Python Spark SQL Hive integration for the EDSTD project") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:1.0.0") \
    .enableHiveSupport() \

spark = spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
spark.sql(
    """
    SHOW TABLES FROM projeto
    """
).show()

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|  projeto|        amazon_books|      false|
|  projeto|      amazon_credits|      false|
|  projeto|       amazon_titles|      false|
|  projeto|    amazon_titles_v2|      false|
|  projeto|    book_movie_adapt|      false|
|  projeto|          box_office|      false|
|  projeto|imdb_movie_adapta...|      false|
|  projeto|imdb_tvshow_adapt...|      false|
|  projeto|     netflix_credits|      false|
|  projeto|      netflix_titles|      false|
|  projeto|   netflix_titles_v2|      false|
|  projeto|   streaming_credits|      false|
|  projeto|    streaming_titles|      false|
+---------+--------------------+-----------+



In [9]:
# spark.sql(
#    """
#    DROP TABLE IF EXISTS projeto.amazon_books
#    """
#)

DataFrame[]

In [10]:
spark.sql("""
    CREATE EXTERNAL TABLE projeto.amazon_books (
        asin VARCHAR(10),
        ISBN10 VARCHAR(15),
        title VARCHAR(250),
        brand VARCHAR(500),
        availability VARCHAR(270),
        discount DOUBLE,
        initial_price DOUBLE,
        final_price DOUBLE,
        currency VARCHAR(5),
        rating VARCHAR(20),
        reviews_count INT,
        categories VARCHAR(130),
        best_sellers_rank VARCHAR(3500),
        url VARCHAR(50)
    )
    USING DELTA
    PARTITIONED BY (rating)
    LOCATION 'hdfs://hdfs-nn:9000/warehouse/projeto.db/amazon_books/'
"""
)

DataFrame[]

In [12]:
spark.sql(
    """
    DESCRIBE FORMATTED projeto.amazon_books
    """
).toPandas()

,col_name,data_type,comment
0,asin,string,None
1,ISBN10,string,None
2,title,string,None
3,brand,string,None
4,availability,string,None
5,discount,double,None
6,initial_price,double,None
7,final_price,double,None
8,currency,string,None
9,rating,string,None


In [16]:
spark.sql(
    """
    SELECT * FROM projeto.amazon_books
    """
).show()

+----------+-------------+--------------------+--------------------+--------------------+--------+-------------+-----------+--------+------------------+-------------+--------------------+--------------------+--------------------+
|      asin|       ISBN10|               title|               brand|        availability|discount|initial_price|final_price|currency|            rating|reviews_count|          categories|   best_sellers_rank|                 url|
+----------+-------------+--------------------+--------------------+--------------------+--------+-------------+-----------+--------+------------------+-------------+--------------------+--------------------+--------------------+
|0062909711|   0062909711|The Plant Paradox...|by Dr. Steven R G...|In stock. Usually...|    null|         null|      26.55|     USD|4.4 out of 5 stars|        12492|["Books","Health,...|[{"category":"Boo...|https://www.amazo...|
|0062963686|   0062963686|The Dutch House: ...|        Ann Patchett|           I

In [128]:
spark.sql(
    """
    DESCRIBE HISTORY projeto.amazon_books
    """
).show()

+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|   operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      1|2025-10-31 21:26:...|  null|    null|       WRITE|{mode -> Overwrit...|null|    null|     null|          0|  Serializable|        false|{numFiles -> 13, ...|        null|Apache-Spark/3.4....|
|      0|2025-10-31 21:26:...|  null|    null|CREATE TABLE|{isManaged -> fal...|null|    null|     null|       null|  Serializable|         true|                  {}|        null|Apache-Spark/3.4.